In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers

In [ ]:
import json
from google.colab import files

# --- Upload your original dialogue.json ---
uploaded = files.upload()  # select dialogue.json
with open("dialogue.json") as f:
    original = json.load(f)

print(f"Original dialogues: {len(original)}")

# --- 40 additional dialogues (fixed for reproducibility) ---
EXTRA_TOPICS = [
    ("robotics", "home robots and daily assistance"),
    ("space", "commercial space travel for ordinary people"),
    ("food", "lab-grown meat and food technology"),
    ("music", "AI-generated music and copyright"),
    ("gaming", "the rise of cloud gaming platforms"),
    ("privacy", "personal data protection in the digital age"),
    ("transport", "autonomous vehicles on public roads"),
    ("energy", "nuclear fusion as a future power source"),
    ("aging", "technology for elderly care and independence"),
    ("language", "how translation AI affects language learning"),
    ("design", "generative AI in product design workflows"),
    ("farming", "precision agriculture and drone technology"),
    ("mental_health", "digital therapy and mental wellness apps"),
    ("housing", "3D-printed houses and affordable construction"),
    ("journalism", "AI-assisted reporting and media trust"),
    ("sports", "data analytics changing team strategy"),
    ("law", "AI tools in legal research and contracts"),
    ("retail", "cashier-free stores and shopping automation"),
    ("water", "smart systems for water conservation"),
    ("tourism", "virtual reality travel experiences"),
    ("supply_chain", "AI-driven logistics and delivery optimization"),
    ("cybersecurity", "defending small businesses from cyber attacks"),
    ("fashion", "sustainable fashion and AI trend prediction"),
    ("construction", "building information modeling and AI planning"),
    ("insurance", "automated claims processing with AI"),
    ("archaeology", "satellite imaging and AI in discovering ruins"),
    ("childcare", "screen time management tools for parents"),
    ("volunteering", "platforms connecting volunteers with local causes"),
    ("accessibility", "AI tools that help visually impaired users"),
    ("sleep", "sleep tracking technology and its real impact"),
    ("pets", "wearable health monitors for pets"),
    ("art", "AI-assisted art restoration in museums"),
    ("waste", "smart recycling systems in urban areas"),
    ("reading", "AI-curated reading lists and book recommendations"),
    ("voting", "digital voting systems and election security"),
    ("fitness", "AI personal trainers and workout planning"),
    ("shipping", "autonomous cargo ships and maritime logistics"),
    ("dentistry", "AI diagnostics in dental imaging"),
    ("libraries", "how public libraries adapt to the digital era"),
    ("comedy", "can AI understand and generate humor"),
]

TEMPLATES = [
    [
        ("host", "Today we are diving into {topic_desc}. What should people know first?"),
        ("expert", "The key starting point is understanding what problem this solves. Most people hear the buzzword but miss the practical value."),
        ("host", "So awareness comes before adoption. What does a realistic first step look like?"),
        ("expert", "Start with a small pilot. Test one specific use case, measure the outcome, and then decide whether to expand."),
    ],
    [
        ("host", "Our topic today is {topic_desc}. Where do you see the biggest opportunity?"),
        ("expert", "The biggest opportunity is in reducing repetitive effort. When people spend less time on routine tasks, they can focus on creative work."),
        ("host", "But there must be challenges as well. What holds people back?"),
        ("expert", "Usually it is a mix of cost, trust, and habit. Change is hard even when the benefits are clear."),
    ],
    [
        ("host", "Let us talk about {topic_desc}. Is the hype justified?"),
        ("expert", "Some of it is. The technology works, but the gap between a demo and real-world deployment is still large."),
        ("host", "What would help close that gap faster?"),
        ("expert", "Better standards, more open data, and honest conversations about limitations. Overpromising slows down real progress."),
    ],
    [
        ("host", "We are exploring {topic_desc} in this episode. Who benefits the most?"),
        ("expert", "Right now, early adopters with technical resources benefit first. But the long-term goal should be broad accessibility."),
        ("host", "How do we make sure it reaches everyone, not just a few?"),
        ("expert", "Public investment, open-source tools, and education programs all play a role. No single solution is enough on its own."),
    ],
]

augmented = list(original)
for i, (topic, desc) in enumerate(EXTRA_TOPICS):
    template = TEMPLATES[i % len(TEMPLATES)]
    turns = [{"speaker": sp, "text": txt.format(topic_desc=desc)} for sp, txt in template]
    augmented.append(
        {
            "dialogue_id": f"dlg_{len(original) + i + 1:03d}",
            "topic": topic,
            "turns": turns,
        }
    )

with open("dialogue_50.json", "w") as f:
    json.dump(augmented, f, indent=2)

print(f"Total dialogues: {len(augmented)}")

In [ ]:
import json

with open("dialogue_50.json") as f:
    dialogues = json.load(f)

sft_data = []
for dlg in dialogues:
    messages = [
        {
            "role": "system",
            "content": (
                f"You are an insightful podcast guest. The topic is {dlg['topic']}. "
                "Give concise, thoughtful answers to the host's questions."
            ),
        }
    ]
    for turn in dlg["turns"]:
        role = "user" if turn["speaker"] == "host" else "assistant"
        messages.append({"role": role, "content": turn["text"]})
    sft_data.append({"messages": messages})

with open("train_sft.json", "w") as f:
    json.dump(sft_data, f, indent=2)

print(f"SFT samples: {len(sft_data)}")
print("Example:")
print(json.dumps(sft_data[0], indent=2))

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import torch

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")

if "A100" in gpu_name:
    MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
    BATCH_SIZE = 4
else:
    MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
    BATCH_SIZE = 2

print(f"Using model: {MODEL_NAME}, batch_size: {BATCH_SIZE}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

dataset = load_dataset("json", data_files="train_sft.json", split="train")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="./output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        num_train_epochs=5,
        warmup_steps=5,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        save_strategy="epoch",
        optim="adamw_8bit",
        seed=42,
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"Training complete. Final loss: {stats.training_loss:.4f}")

In [ ]:
from unsloth import FastLanguageModel
from google.colab import files

FastLanguageModel.for_inference(model)

test_prompts = [
    ("climate", "What is the most overlooked aspect of climate change policy?"),
    ("ai", "How should universities teach AI ethics to engineering students?"),
    ("media", "Do you think traditional newspapers will survive another decade?"),
]

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)

for topic, question in test_prompts:
    messages = [
        {
            "role": "system",
            "content": (
                f"You are an insightful podcast guest. The topic is {topic}. "
                "Give concise, thoughtful answers to the host's questions."
            ),
        },
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to("cuda")

    outputs = model.generate(
        inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\n[Topic: {topic}]")
    print(f"Host: {question}")
    print(f"Expert: {response}")
    print("-" * 60)

model.save_pretrained("podcast-expert-lora")
tokenizer.save_pretrained("podcast-expert-lora")

!zip -r podcast-expert-lora.zip podcast-expert-lora/
files.download("podcast-expert-lora.zip")

print("\nAdapter saved and ready for download.")